# 03 - Degrade and augment (CCTV-style)

Turns the **undegraded** images in `data/synthetic_clean/{class}/` into the **degraded** training set
in `data/synthetic_degraded/{class}/`, simulating real restaurant-camera footage: low resolution,
blur, sensor noise, lighting/exposure, white-balance cast, vignetting, JPEG/codec blocking and a
mounted-camera viewpoint. This is **novelty #1**.

**3-class project** (`clean`, `finished`, `full`) - the notebook reads whatever class folders exist on
disk, so it adapts automatically (no class list hard-coded).

### What changed vs. the original (and why)
- **De-biased: the degradation is class-AGNOSTIC.** `degrade_image(img)` never sees the label, so no
  degradation parameter can correlate with the class. The old code gave `clean` *less* noise than the
  other classes - a shortcut the model could exploit ("low noise => clean") that breaks on real CCTV.
- **Less, unified noise.** One noise distribution for every class, and lower overall (`std 3-10` instead
  of `5-15`) so images stay readable. (Note: step 05's train transform adds a little more noise on top.)
- **Softer borders.** Tilt/perspective use edge-replication instead of black fill or mirroring, so
  rotation/warp don't paint tell-tale black corners the model could key on.
- **Reproducible.** Per-image augmentation seeds are derived deterministically from the filename
  (the old code re-seeded *randomly* each image, violating the project's fixed-seed rule).
- **JPEG via an in-memory buffer** (no shared `temp.jpg` on disk - safe with parallel workers).
- **Optional vignetting** (cheap-lens corner darkening) for extra CCTV realism.

Keep both copies on disk (clean = undegraded, degraded) so the with/without-degradation ablation in
step 05 can run. No timestamp / "REC" / camera-id HUD is ever drawn - that would be a shortcut cue.

## Configuration + paths

In [ ]:
import os, random, shutil, hashlib
from io import BytesIO
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import cv2

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

random.seed(utils.SEED); np.random.seed(utils.SEED)

# --- knobs ---
INPUT_DIR  = utils.CLEAN_DIR          # data/synthetic_clean/{class}/  (point elsewhere if needed)
OUTPUT_DIR = utils.DEGRADED_DIR       # data/synthetic_degraded/{class}/
NUM_AUG_CHOICES = [2, 3, 4, 5]        # how many degraded copies per source image
USE_90_ROTATIONS = False              # 90/180/270 flips are a rotation-invariance aug, NOT a CCTV
                                      # artifact (a flipped angled plate looks unnatural). Off by default.

# Classes = whatever subfolders actually exist on disk (e.g. clean / finished / full).
CLASSES = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()]) if INPUT_DIR.is_dir() else []
print("input  :", INPUT_DIR)
print("output :", OUTPUT_DIR)
print("classes:", CLASSES)

def reset_degraded_folder():
    if OUTPUT_DIR.exists():
        print("Deleting existing degraded folder..."); shutil.rmtree(OUTPUT_DIR)
    for cls in CLASSES:
        os.makedirs(OUTPUT_DIR / cls, exist_ok=True)
    print("Fresh degraded folder ready:", OUTPUT_DIR)

## Degradation primitives

Each is a small, self-contained CCTV effect. Borders use `BORDER_REPLICATE` (repeat the edge pixel)
rather than black/mirror, so geometric transforms don't introduce artificial corner cues.

In [ ]:
def add_blur(img, radius_range=(0.3, 1.0)):
    return img.filter(ImageFilter.GaussianBlur(random.uniform(*radius_range)))

def add_noise(img, std_range=(3, 10)):
    # Gaussian sensor noise. SAME range for every class (no per-class bias).
    arr = np.array(img).astype(np.float32)
    arr += np.random.normal(0, random.uniform(*std_range), arr.shape)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def change_lighting(img, brightness_range=(0.7, 1.4), contrast_range=(0.7, 1.4)):
    img = ImageEnhance.Brightness(img).enhance(random.uniform(*brightness_range))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(*contrast_range))
    return img

def downscale(img, sizes=(112, 128, 160, 192)):
    # The defining CCTV cue: drop resolution then upscale back. (Dropped the harsh 96px floor.)
    size = random.choice(sizes)
    return img.resize((size, size), Image.BILINEAR).resize((224, 224), Image.BILINEAR)

def compress(img, quality_range=(30, 70)):
    # JPEG/codec blocking via an in-memory buffer (no shared temp file on disk).
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=random.randint(*quality_range))
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def color_shift(img, shift_range=(0.9, 1.12)):
    # Per-channel gain = white-balance / colour cast of a cheap camera.
    arr = np.array(img).astype(np.float32)
    for ch in range(3):
        arr[:, :, ch] *= random.uniform(*shift_range)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def add_vignette(img, strength_range=(0.15, 0.4)):
    # Cheap-lens corner darkening.
    arr = np.array(img).astype(np.float32)
    h, w = arr.shape[:2]
    strength = random.uniform(*strength_range)
    yy = np.linspace(-1, 1, h)[:, None]
    xx = np.linspace(-1, 1, w)[None, :]
    r = np.sqrt(xx**2 + yy**2) / np.sqrt(2.0)
    mask = (1.0 - strength * (r ** 2))[:, :, None]
    return Image.fromarray(np.clip(arr * mask, 0, 255).astype(np.uint8))

def rotate_image(img):
    # 90/180/270 invariance aug (only used if USE_90_ROTATIONS).
    return img.rotate(random.choice([0, 90, 180, 270]), expand=True)

def slight_rotate(img, angle_range=(-15, 15)):
    # Small tilt with edge replication (no black corners).
    arr = np.array(img); h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), random.uniform(*angle_range), 1.0)
    out = cv2.warpAffine(arr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(out)

def perspective_transform(img, scale_range=(0.05, 0.15)):
    # Mounted-camera viewpoint: warp the plane, replicate the border.
    w, h = img.size
    shift = random.uniform(*scale_range) * w
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([
        [random.uniform(0, shift), random.uniform(0, shift)],
        [w - random.uniform(0, shift), random.uniform(0, shift)],
        [w - random.uniform(0, shift), h - random.uniform(0, shift)],
        [random.uniform(0, shift), h - random.uniform(0, shift)]])
    warped = cv2.warpPerspective(np.array(img), cv2.getPerspectiveTransform(src, dst),
                                 (w, h), borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(warped)

## The degradation pipeline

Per-image randomized, with a `severity` roll so some images are lightly degraded and some heavily
(as in real footage). **Class-agnostic:** `degrade_image` takes only the image - it cannot bias by
label.

In [ ]:
def degrade_image(img):
    severity = random.random()                      # per-image overall intensity

    # 1) Low resolution - the dominant CCTV cue (always applied)
    img = downscale(img)

    # 2) Geometry: mild tilt + mounted-camera perspective
    if random.random() < 0.7:
        img = slight_rotate(img, (-8, 8) if severity < 0.5 else (-15, 15))
    if USE_90_ROTATIONS and random.random() < 0.5:
        img = rotate_image(img)
    if random.random() < 0.6:
        img = perspective_transform(img, (0.05, 0.12) if severity < 0.5 else (0.12, 0.22))

    # 3) Blur (defocus)
    if random.random() < 0.6:
        img = add_blur(img, (0.3, 1.0) if severity < 0.5 else (1.0, 2.0))

    # 4) Sensor noise - SAME distribution for every class (de-biased)
    if random.random() < 0.6:
        img = add_noise(img, (3, 10))

    # 5) Lighting / exposure
    if random.random() < 0.6:
        img = change_lighting(img, (0.7, 1.4), (0.7, 1.4))

    # 6) White-balance / colour cast
    if random.random() < 0.4:
        img = color_shift(img, (0.9, 1.12))

    # 7) Vignetting (cheap-lens corner darkening)
    if random.random() < 0.35:
        img = add_vignette(img, (0.15, 0.4))

    # 8) JPEG / codec blocking artifacts
    if random.random() < 0.6:
        img = compress(img, (30, 70))

    return img.convert("RGB")

## Generate the degraded dataset

2-5 degraded copies per source image. Augmentation seeds are derived **deterministically** from each
filename, so the degraded set is fully reproducible.

In [ ]:
from tqdm import tqdm

def _stable_seed(stem, i):
    """Deterministic per (image, aug-index) seed - reproducible across runs/machines."""
    h = int(hashlib.md5(stem.encode("utf-8")).hexdigest(), 16)
    return (utils.SEED + (h % 1_000_000) + i) % (2**32)

def generate_augmented_dataset():
    reset_degraded_folder()
    exts = (".jpg", ".jpeg", ".png")
    for cls in CLASSES:
        in_dir, out_dir = INPUT_DIR / cls, OUTPUT_DIR / cls
        images = sorted(p for p in in_dir.glob("*") if p.suffix.lower() in exts)
        print(f"\n{cls}: {len(images)} source images")
        for img_path in tqdm(images):
            img = Image.open(img_path).convert("RGB")
            random.seed(_stable_seed(img_path.stem, 0))   # deterministic aug count
            n_aug = random.choice(NUM_AUG_CHOICES)
            for i in range(n_aug):
                s = _stable_seed(img_path.stem, i)
                random.seed(s); np.random.seed(s)
                degrade_image(img.copy()).save(out_dir / f"{img_path.stem}_aug{i}.jpg", quality=95)

generate_augmented_dataset()
print("\nDONE.")
for cls in CLASSES:
    print(cls, len(list((OUTPUT_DIR / cls).glob('*'))))

## Balance classes (upsample the smaller classes to the largest)

In [ ]:
def upsample_to_max():
    random.seed(utils.SEED); np.random.seed(utils.SEED)   # reproducible top-up
    sizes = {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES}
    target = max(sizes.values())
    print("before:", sizes, "| target:", target)
    for cls in CLASSES:
        files = list((OUTPUT_DIR / cls).glob('*'))
        idx = 0
        while len(files) < target:
            src = random.choice(files)
            new = OUTPUT_DIR / cls / f"{src.stem}_extra{idx}.jpg"
            degrade_image(Image.open(src).convert("RGB")).save(new, quality=95)
            files.append(new); idx += 1
    print("after :", {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES})

upsample_to_max()

## Visual check: clean (top row) vs degraded (bottom row) per class

In [ ]:
import matplotlib.pyplot as plt
n = 8
fig, axes = plt.subplots(2 * len(CLASSES), n, figsize=(2 * n, 2 * 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    clean_files = sorted((INPUT_DIR / cls).glob('*'))[:n]
    for i, cp in enumerate(clean_files):
        axes[2*r, i].imshow(Image.open(cp)); axes[2*r, i].axis('off')
        augs = list((OUTPUT_DIR / cls).glob(f"{cp.stem}_aug*.jpg"))
        ax = axes[2*r+1, i]
        ax.imshow(Image.open(random.choice(augs)) if augs else Image.open(cp)); ax.axis('off')
    axes[2*r, 0].set_ylabel(cls, fontsize=12)
plt.tight_layout(); plt.show()